In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
url = "https://www.allocine.fr/film/fichefilm-114782/casting/"
html = requests.get(url)
soup = BeautifulSoup(html.text, 'html.parser')
# Nous filtrons maintenant uniquement sur les acteurs en selectionnant le bon encadré:
soup_director = soup.find('section', {"class" : "section casting-actor"})
# Puis nous cherchons tous les liens contenus dans cet encadré :
links = soup_director.find_all('a')

In [46]:
print(type(soup_director))

<class 'bs4.element.Tag'>


In [ ]:
# Je parcours la liste des liens pour chaque acteur et j'affiche le lien
for i in range (0, len(links)):
    print(links[i])

<a class="meta-title-link" href="/personne/fichepersonne_gen_cpersonne=19334.html">Matthew McConaughey</a>
<a class="meta-title-link" href="/personne/fichepersonne_gen_cpersonne=65719.html">Anne Hathaway</a>
<a class="meta-title-link" href="/personne/fichepersonne_gen_cpersonne=2535.html">Michael Caine</a>
<a class="meta-title-link" href="/personne/fichepersonne_gen_cpersonne=5293.html">John Lithgow</a>
<a class="meta-title-link" href="/personne/fichepersonne_gen_cpersonne=117304.html">Jessica Chastain</a>
<a class="meta-title-link" href="/personne/fichepersonne_gen_cpersonne=1196.html">Casey Affleck</a>
<a class="meta-title-link" href="/personne/fichepersonne_gen_cpersonne=487728.html">Mackenzie Foy</a>
<a class="meta-title-link" href="/personne/fichepersonne_gen_cpersonne=35000.html">Wes Bentley</a>
<a class="item link" href="/personne/fichepersonne_gen_cpersonne=205411.html" title="David Gyasi Romilly">David Gyasi</a>
<a class="item link" href="/personne/fichepersonne_gen_cpersonne=

In [6]:
# J'affiche le href du premier lien
print(links[0]['href'])

# J'affiche le nom qui correspond au premier lien
print(links[0].text)


/personne/fichepersonne_gen_cpersonne=19334.html
Matthew McConaughey


In [ ]:
# Je crée un dataframe à partir de la liste des liens

df_url = pd.DataFrame({"name" : [i.text for i in links],
              "href" : [i["href"] for i in links]})
df_url

,name,href
0,Matthew McConaughey,/personne/fichepersonne_gen_cpersonne=19334.html
1,Anne Hathaway,/personne/fichepersonne_gen_cpersonne=65719.html
2,Michael Caine,/personne/fichepersonne_gen_cpersonne=2535.html
3,John Lithgow,/personne/fichepersonne_gen_cpersonne=5293.html
4,Jessica Chastain,/personne/fichepersonne_gen_cpersonne=117304.html
5,Casey Affleck,/personne/fichepersonne_gen_cpersonne=1196.html
6,Mackenzie Foy,/personne/fichepersonne_gen_cpersonne=487728.html
7,Wes Bentley,/personne/fichepersonne_gen_cpersonne=35000.html
8,David Gyasi,/personne/fichepersonne_gen_cpersonne=205411.html
9,Timothée Chalamet,/personne/fichepersonne_gen_cpersonne=525421.html


In [22]:
# Nous récuperons maintenant les liens entiers en ajoutant le nom de domaine devant le href

url = "https://www.allocine.fr/" + "/personne/fichepersonne_gen_cpersonne=19334.html"
html = requests.get(url)
soup = BeautifulSoup(html.text, 'html.parser')
detail = soup.find_all("div", {"class" : "dark-grey"})
detail

[<div class="dark-grey">Américain</div>,
 <div class="dark-grey"><strong>56</strong> ans</div>]

In [29]:
# Nous passons par regex pour récupérer l'age 

import re
age_liste = re.findall("(\d+)", str(detail))
age_liste

['56']

In [31]:
# J'extrais l'age de ma liste, c'est un str donc il faut le changer en integer pour pouvoir faire une moyenne après

age_str = age_liste[0]
age_str

'56'

In [34]:
# Je transforme mon age en entier

age = int(age_str)
print(f"L'age est {age} ans pour Matthew McConaughey et le type de l'age est bien un {type(age)}")

L'age est 56 ans pour Matthew McConaughey et le type de l'age est bien un <class 'int'>


In [ ]:
# Je fais une fonction pour regrouper toutes les étapes ci dessus

def crawl(url_short):
    url = "https://www.allocine.fr" + url_short
    html = requests.get(url)
    soup = BeautifulSoup(html.text, 'html.parser')
    detail = soup.find_all("div", {"class" : "dark-grey"})
    age_liste = re.findall("(\d+)", str(detail))
    if age_liste:
        age_str = age_liste[0]
        age = int(age_str)
        return age - 12 # Je soustrais 12 ans pour avoir l'age de l'acteur au moment du tournage du film
    return None

In [60]:
# Ma fonction marche bien sur la première ligne de ma colonne href, l'age est bien 56

crawl(df_url['href'][0])

44

In [61]:
# Je crée une colonne avec l'age des acteurs en appliquant ma fonction à la colonne href

df_url['age'] = df_url['href'].apply(crawl)

In [62]:
# J'affiche mon df

df_url

,name,href,age
0,Matthew McConaughey,/personne/fichepersonne_gen_cpersonne=19334.html,44.0
1,Anne Hathaway,/personne/fichepersonne_gen_cpersonne=65719.html,31.0
2,Michael Caine,/personne/fichepersonne_gen_cpersonne=2535.html,81.0
3,John Lithgow,/personne/fichepersonne_gen_cpersonne=5293.html,68.0
4,Jessica Chastain,/personne/fichepersonne_gen_cpersonne=117304.html,37.0
5,Casey Affleck,/personne/fichepersonne_gen_cpersonne=1196.html,38.0
6,Mackenzie Foy,/personne/fichepersonne_gen_cpersonne=487728.html,13.0
7,Wes Bentley,/personne/fichepersonne_gen_cpersonne=35000.html,35.0
8,David Gyasi,/personne/fichepersonne_gen_cpersonne=205411.html,34.0
9,Timothée Chalamet,/personne/fichepersonne_gen_cpersonne=525421.html,18.0


In [65]:
# Je calcule la moyenne d'age des acteurs avec un .mean()

print(f"La moyenne d'age des acteurs au moment de la sortie est de {df_url['age'].mean().round(1)} ans pour le film Interstellar")

La moyenne d'age des acteurs au moment de la sortie est de 43.4 ans pour le film Interstellar
